# 🚀 Stack 2.9 - Colab Training Notebook

**Zero-cost training on Google Colab free tier**

This notebook trains a LoRA adapter for Stack 2.9 Pattern Memory on **Qwen2.5-Coder-7B** using a free T4 GPU.

⏱️ **Expected runtime:** 3-5 hours
💾 **VRAM needed:** ~12GB (fits in T4's 15GB)
📦 **Output:** `./adapters_colab/` (LoRA) + `./model_final/` (merged)

---

**Instructions:**
1. Runtime → Change runtime type → **GPU (T4)**
2. Run each cell in order (Shift+Enter or Play button)
3. Monitor progress in cell outputs

---

In [34]:
# Check GPU availability
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## 1️⃣ Mount Google Drive (Optional)

Mount Drive to persist data between sessions. If you skip this, data is stored in Colab's ephemeral storage (lost after ~12h or disconnect).

In [24]:
from google.colab import drive
drive.mount('/content/drive')

# Set base path (change if not using Drive)
BASE_PATH = "/content/drive/MyDrive/stack-2.9"  # or use "/content/stack-2.9" for local storage

print(f"Using base path: {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using base path: /content/drive/MyDrive/stack-2.9


## 2️⃣ Clone Repository & Install Dependencies

In [25]:
import os
os.chdir('/content')

# Clone the Stack 2.9 repository if not already present
if not os.path.exists('stack-2.9'):
    !git clone https://github.com/my-ai-stack/stack-2.9.git

os.chdir('/content/stack-2.9')

# Upgrade pip and install core dependencies
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.40.0 peft==0.10.0 accelerate bitsandbytes==0.43.0 datasets pyyaml

# Verify bitsandbytes GPU support (critical for 4-bit training)
import bitsandbytes as bnb
try:
    import torch
    if torch.cuda.is_available():
        print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
        # Check if bitsandbytes can find the GPU
        from bitsandbytes.cextension import COMPILED_WITH_CUDA
        if COMPILED_WITH_CUDA:
            print("✅ bitsandbytes successfully compiled with CUDA support.")
        else:
            print("⚠️ bitsandbytes compiled WITHOUT CUDA support. Training will fail.")
    else:
        print("❌ No GPU detected. Please change runtime to T4 GPU.")
except Exception as e:
    print(f"❌ Error checking GPU status: {e}")

# Install specific training requirements
if os.path.exists('stack-2.9-training/requirements.txt'):
    !pip install -r stack-2.9-training/requirements.txt

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 13.4 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement awq>=0.1.0 (from versions: none)
ERROR: No matching distribution found for awq>=0.1.0


## 3️⃣ Prepare Training Data

### Option A: Use existing training data from repository
The repo already has `training-data/final/train.jsonl` and `eval.jsonl` if you previously ran data collection.

### Option B: Create a mini dataset for quick prototyping (5K examples)
Recommended for first run to verify everything works quickly.

In [26]:
# Robust fix for missing imports in the mini dataset script
# Uses pattern matching to insert after the first line (shebang or existing imports)
!sed -i '1a from typing import List, Dict, Optional, Any' scripts/create_mini_dataset.py
print("✅ Patch applied to scripts/create_mini_dataset.py")

In [27]:
# Create mini dataset (5K examples)
!python scripts/create_mini_dataset.py --size 5000 --output data_mini/train_mini.jsonl --source training-data/final/train.jsonl

# Check what we have
!ls -lh data_mini/

# If you want to use the full dataset instead, skip the mini creation and use:
# training-data/final/train.jsonl (and eval.jsonl if available)

Loading full dataset from training-data/final/train.jsonl...
Loaded 41807 total examples

Tool distribution in full dataset:
  No-tool examples: 41807 (100.0%)

Sampling plan (target 5000):
  __notool__: 5000 examples (100.0%) from 41807 available

✅ Mini dataset created: 5000 examples
   Saved to: data_mini/train_mini.jsonl

Final tool distribution:
  __notool__: 5000 (100.0%)
total 2.4M
-rw-r--r-- 1 root root 2.4M Apr  3 13:59 train_mini.jsonl


## 4️⃣ Prepare Training Configuration

Copy the Colab-optimized config (7B model, 8K seq len, 2 epochs).

In [28]:
# Copy the Colab config or create one manually
!cp stack_2_9_training/train_config_colab.yaml stack_2_9_training/train_config.yaml

# If you need to use your own data, edit the paths in train_config.yaml:
# data:
#   train_file: "./data_mini/train_mini.jsonl"  # or ./training-data/final/train.jsonl
#   validation_file: "./training-data/final/eval.jsonl"  # optional

print("Configuration ready. Showing relevant sections:")
with open('stack_2_9_training/train_config.yaml', 'r') as f:
    lines = f.readlines()
    for i, line in enumerate(lines[:50]):  # Show first 50 lines
        print(f"{i+1}: {line.rstrip()}")
print("...")

Configuration ready. Showing relevant sections:
1: # Colab-Optimized Training Configuration for Stack 2.9
2: # Target: Google Colab free tier (T4 GPU, 15GB VRAM)
3: # Model: Qwen/Qwen2.5-Coder-7B (4-bit quantized fits in ~4.5GB)
4: # Expected runtime: 3-5 hours
5: 
6: model:
7:   name: "Qwen/Qwen2.5-Coder-7B"  # 7B instead of 32B for Colab
8:   trust_remote_code: true
9:   use_flash_attention: false  # T4 doesn't support flash attention well
10: 
11: tokenizer:
12:   model_max_length: 8192  # Reduced from 131072 for memory
13:   padding_side: "right"
14:   truncation_side: "right"
15: 
16: peft:
17:   peft_type: "LORA"
18:   task_type: "CAUSAL_LM"
19:   r: 16  # LoRA rank (lower = faster, good enough for 7B)
20:   lora_alpha: 32
21:   lora_dropout: 0.05
22:   target_modules:
23:     - "q_proj"
24:     - "k_proj"
25:     - "v_proj"
26:     - "o_proj"
27:     - "gate_proj"
28:     - "up_proj"
29:     - "down_proj"
30:     # Optional: add "embed_tokens", "lm_head" for full coverage (incre

In [29]:
# Update config to point to mini dataset using absolute paths
import os
base_dir = os.getcwd()
train_path = os.path.join(base_dir, "data_mini/train_mini.jsonl")
# Fixed path to point to the correct subdirectory
config_path = "stack-2.9-training/train_config.yaml"

# Ensure we are starting with the repository's default config
!cp stack-2.9-training/train_config_colab.yaml {config_path}

!sed -i 's|train_file: .*|train_file: "{train_path}"|' {config_path}
!sed -i 's|validation_file: .*|# validation_file: not needed|' {config_path}

print(f"✅ Config refreshed and updated at: {config_path}")
print(f"Target training data: {train_path}")

# Verification: Show the updated lines
print("\nUpdated config snippet:")
!grep -E "train_file|validation_file" {config_path}

✅ Config updated with path: /content/stack-2.9/data_mini/train_mini.jsonl

Data section of config:
data:
  input_path: "/Users/walidsobhi/.openclaw/workspace/stack-2.9/training-data/synthetic/examples.jsonl"
  train_dir: "/Users/walidsobhi/.openclaw/workspace/stack-2.9-training/data/train"
  eval_dir: "/Users/walidsobhi/.openclaw/workspace/stack-2.9-training/data/eval"


## 5️⃣ Train LoRA Adapter

This is the main training step. Monitor GPU memory with `nvidia-smi` in a separate terminal or cell.

⚠️ **Training will take 3-5 hours**. Do not interrupt unless necessary.

Training progress is shown with loss values. Lower loss = better learning.

In [30]:
# Start training using the correct directory path
import os
os.environ['PYTHONUNBUFFERED'] = '1'

!python3 stack-2.9-training/train_lora.py --config stack-2.9-training/train_config.yaml

2026-04-03 14:00:07.564139: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775224807.644408   38357 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775224807.658917   38357 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775224807.716959   38357 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775224807.717056   38357 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775224807.717065   38357 computation_placer.cc:177] computation placer alr

## 6️⃣ Verify Training Output

In [31]:
!ls -lh adapters_colab/

# If training succeeded, you should see:
# - adapter_model.bin (or multiple checkpoint-XXX folders)
# - training_args.bin
# - config.json

ls: cannot access 'adapters_colab/': No such file or directory


## 7️⃣ Merge LoRA Adapter with Base Model

Combines the trained adapter weights with the base Qwen2.5-Coder-7B model to produce a standalone fine-tuned model.

In [32]:
# Merge LoRA Adapter with Base Model
# Added missing --lora and --output arguments
!python3 stack-2.9-training/merge_adapter.py \
    --base-model Qwen/Qwen2.5-Coder-7B \
    --lora ./adapters_colab \
    --output ./model_final

print("\n✅ Merged model process completed.")
if os.path.exists('./model_final'):
    !ls -lh model_final/
else:
    print("❌ Output directory ./model_final/ was not created. Check merge logs above.")

usage: merge_adapter.py [-h] [--config CONFIG] [--lora LORA] [--output OUTPUT]
                        [--awq]
merge_adapter.py: error: unrecognized arguments: --base-model Qwen/Qwen2.5-Coder-7B

✅ Merged model created in ./model_final/
ls: cannot access 'model_final/': No such file or directory


## 8️⃣ Test Inference

Quick sanity check: does the model generate reasonable code?

In [33]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os

# Load merged model from the local directory
model_path = "./model_final"

if not os.path.exists(model_path):
    print(f"❌ Error: Local model path {model_path} does not exist. Please run the merge cell first.")
else:
    print(f"Loading model from {model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    # Test generation
    prompt = "Write a Python function to calculate factorial recursively:\n\n```python\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    print("Generating...")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("="*40)
    print("RESPONSE:")
    print("="*40)
    print(response[len(prompt):])

OSError: Incorrect path_or_model_id: './model_final'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

## 9️⃣ Export to Hugging Face Hub (Optional)

If you want to publish your model, push it to Hugging Face and then apply to Together AI.

In [ ]:
from huggingface_hub import HfApi

# You need a Hugging Face account and token
HF_TOKEN = input("Enter your Hugging Face token: ").strip()

api = HfApi(token=HF_TOKEN)

# Choose a repo name
repo_id = input("Enter repository name (e.g., your-org/stack-2.9-7b-lora): ").strip()

print(f"\nUploading to {repo_id}...")

# Create repo if needed
api.create_repo(repo_id=repo_id, exist_ok=True)

# Upload model
api.upload_folder(
    folder_path="./model_final",
    repo_id=repo_id,
    repo_type="model"
)

print(f"\n✅ Model uploaded to https://huggingface.co/{repo_id}")

# Update docs
print("\nNext steps:")
print("1. Update TOGETHER_AI.md with your model ID")
print("2. Update README.md badges with real scores after evaluation")
print("3. Submit to Together AI model submission form")

## 🎉 Training Complete!

You now have:
- ✅ Trained LoRA adapter in `./adapters_colab/`
- ✅ Merged full model in `./model_final/`
- ✅ Model card and documentation

**Next steps:**
1. Run proper evaluation using `run_proper_evaluation.py`
2. Update README with real benchmark scores
3. Apply to Together AI with your Hugging Face model

**Need help?** See `COLAB_TRAINING.md` for detailed troubleshooting.